# Introdução Suave ao PyTorch

> Parte da série [ML Notebooks](../README.md) — por **Nandobez**.



Este é um tutorial curto, sem jargão e sem código complicado. Mesmo assim, é talvez um dos modelos mais básicos que se pode construir com PyTorch.

De fato, é tão básico que é ideal para quem está começando a aprender PyTorch e deep learning. Então, se você tem um amigo ou colega que quer dar o primeiro passo, recomende este tutorial como ponto de partida. Vamos começar!


---


## Primeiros Passos

Precisamos importar alguns módulos úteis para conseguir as funções necessárias para construir nosso modelo de deep learning. Os principais são `torch` e `torchvision`, que contêm a maior parte das funções para começar com PyTorch. Como este é um tutorial de deep learning, também usaremos `torch.nn`, `torch.nn.functional` e `torchvision.transforms`, que trazem utilidades para construir o modelo. Provavelmente não vamos usar todos os módulos listados abaixo, mas eles são os típicos que se importa ao iniciar um projeto de deep learning.


In [ ]:
## The usual imports
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

## for printing image
import matplotlib.pyplot as plt
import numpy as np

## Carregando os Dados

Vamos direto ao ponto. Como em qualquer projeto de aprendizado de máquina, é preciso carregar o conjunto de dados. Vamos usar o [conjunto MNIST](http://yann.lecun.com/exdb/mnist/), o "Olá Mundo" dos conjuntos de dados em aprendizado de máquina.

Os dados são imagens de tamanho `28 × 28`. Vamos discutir as imagens em breve, mas a ideia é carregar os dados em batches de tamanho `32`.

Aqui estão os passos completos ao importar os dados:

- Importamos e transformamos os dados em tensores usando o módulo `transforms`.
- Usamos `DataLoader` para construir loaders convenientes, que facilitam alimentar batches no modelo de forma eficiente. Vamos falar de batches mais à frente — por ora, pense neles como subconjuntos dos dados.
- Como dito, criamos batches definindo o parâmetro `batch` dentro do DataLoader. Note que usamos batches de `32` neste tutorial, mas você pode mudar para `64` se preferir.


In [ ]:
## parameter denoting the batch size
BATCH_SIZE = 32

## transformations
transform = transforms.Compose(
    [transforms.ToTensor()])

## download and load training dataset
trainset = torchvision.datasets.MNIST(root='./data', train=True,
                                        download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=BATCH_SIZE,
                                          shuffle=True, num_workers=2)

## download and load testing dataset
testset = torchvision.datasets.MNIST(root='./data', train=False,
                                       download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=BATCH_SIZE,
                                         shuffle=False, num_workers=2)

Vamos inspecionar o que os objetos `trainset` e `testset` contêm.


In [ ]:
## print the trainset and testset
print(trainset)
print(testset)

Este é um tutorial para iniciantes, então vou destrinchar:

- `BATCH_SIZE` é o parâmetro que indica o tamanho do batch que vamos usar.
- `transform` guarda o código das transformações aplicadas aos dados. Mais à frente vou mostrar um exemplo do que isso faz na prática.
- `trainset` e `testset` contêm os objetos de conjunto de dados de fato. Note que uso `treinar=True` para indicar o conjunto de treinamento, e `treinar=False` para o restante (o conjunto de teste). Pela porção impressa acima, dá pra ver que a divisão foi 85% (60000) / 15% (10000), correspondente a treino e teste.
- `trainloader` é o DataLoader, que cuida de embaralhar os dados e construir os batches.


Agora vamos olhar a função `transforms.Compose(...)` e ver o que ela faz. Usamos uma imagem aleatória para demonstrar. Vamos gerar essa imagem.


In [ ]:
image = transforms.ToPILImage(mode='L')(torch.randn(1, 96, 96))

**E** vamos renderizá-la:


In [ ]:
plt.imshow(image)

Pronto, temos nossa imagem de exemplo. Agora vamos aplicar uma transformação simples nela: vamos girar a imagem em `45` graus. A transformação abaixo cuida disso.


In [ ]:
## dummy transformation
dummy_transform = transforms.Compose(
    [transforms.RandomRotation(45)])

dummy_result = dummy_transform(image)

plt.imshow(dummy_result)

Note que você pode colocar várias transformações dentro de `transforms.Compose(...)`. Pode usar transformações nativas do PyTorch ou criar as suas próprias e compor como quiser. De fato, você pode encadear quantas transformações precisar. Vamos tentar outra composição: rotação + flip vertical.


In [ ]:
## dummy transform 
dummy2_transform = transforms.Compose(
    [transforms.RandomRotation(45), transforms.RandomVerticalFlip()])

dummy2_result = dummy2_transform(image)

plt.imshow(dummy2_result)

Bem legal, né? Continue testando outros métodos de transformação. Falando em explorar os dados, vamos dar uma olhada nas imagens do nosso conjunto de dados.


## Explorando os Dados

Como praticante e pesquisador, sempre gasto um tempo explorando e entendendo meus conjuntos de dados. É divertido e é uma boa prática para garantir que está tudo em ordem.

Vamos ver o que o conjunto de treino e o de teste contêm. Uso `matplotlib` para mostrar algumas imagens. Com um pouco de NumPy posso convertê-las para arrays e exibi-las. Abaixo mostro o batch inteiro.


In [ ]:
## functions to show an image
def imshow(img):
    #img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)))

## get some random training images
dataiter = iter(trainloader)
images, labels = dataiter.next()

## show images
imshow(torchvision.utils.make_grid(images))

As dimensões dos nossos batches são:


In [ ]:
for images, labels in trainloader:
    print("Image batch dimensions:", images.shape)
    print("Image label dimensions:", labels.shape)
    break

## Modelo

Agora vamos construir um modelo de deep learning para classificar imagens. Vamos manter simples: empilhamos algumas camadas densas e uma camada de dropout para treinar o modelo.

Vamos discutir o modelo:

- Antes de tudo, a estrutura com `class` abaixo é o código padrão para construir um modelo de rede neural em PyTorch:

```python
class MeuModelo(nn.Module):
    def __init__(self):
        super().__init__()
        # camadas aqui

    def forward(self, x):
        # cálculos aqui
```

- As camadas são definidas dentro de `__init__()`. `super().__init__()` apenas amarra as coisas. No nosso modelo, empilhamos uma camada oculta (`self.d1`), seguida de uma camada de dropout (`self.dropout`), seguida da camada de saída (`self.d2`).
- `nn.Linear(...)` define uma camada densa e precisa das dimensões `in` e `out`, correspondentes ao tamanho da feature de entrada e de saída.
- `nn.Dropout(...)` define uma camada de dropout, técnica de regularização que ajuda a evitar overfitting. Queremos que o modelo generalize bem para exemplos não vistos. O dropout zera aleatoriamente algumas unidades com probabilidade `p=0.2`. Mais detalhes [aqui](https://pytorch.org/docs/stable/nn.html#dropout).
- O ponto de entrada do modelo (onde os dados chegam) fica em `forward(...)`. Tipicamente, também colocamos ali outras transformações aplicadas durante o treinamento.
- No `forward()` realizamos uma série de operações:
  - achatamos a imagem, convertendo de 2D (`28 × 28`) para 1D (`1 × 784`)
  - alimentamos os batches dessas imagens 1D na primeira camada oculta
  - aplicamos a [função de ativação não linear](https://en.wikipedia.org/wiki/Rectifier_(neural_networks)) `ReLU`. O importante é que ela permite treinar arquiteturas neurais maiores e mais rápido em conjuntos grandes.
  - o dropout ajuda a evitar overfitting nos dados de treino
  - passamos a saída do dropout para a camada de saída (`d2`)
  - o resultado vai para [softmax](https://en.wikipedia.org/wiki/Softmax_function), que normaliza a saída em uma distribuição de probabilidade para gerar predições válidas — usadas para calcular a acurácia.


In [ ]:
## the model
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.d1 = nn.Linear(28 * 28, 128)
        self.dropout = nn.Dropout(p=0.2)
        self.d2 = nn.Linear(128, 10)
    
    def forward(self, x):
        x = x.flatten(start_dim = 1)
        x = self.d1(x)
        x = F.relu(x)
        x = self.dropout(x)
        logits = self.d2(x)
        out = F.softmax(logits, dim=1)
        return out

Visualmente, abaixo está o diagrama do modelo que construímos. A camada oculta é bem maior do que aparece — o desenho é só uma aproximação por questão de espaço.


Como nos meus tutoriais anteriores, sempre recomendo testar o modelo com 1 batch para garantir que as dimensões de saída batem com o esperado. Note como iteramos sobre o `dataloader`, que entrega pares `images` / `labels`. `out` contém a saída do modelo — logits que passam pelo softmax para gerar predições.


In [ ]:
## test the model with 1 batch
model = MyModel()
for images, labels in trainloader:
    print("batch size:", images.shape)
    out = model(images)
    print(out.shape)
    break

Dá para ver claramente que recebemos batches com 10 valores de saída associados. Esses valores são usados para calcular a performance do modelo.


## Treinando o Modelo

Pronto para treinar. Antes disso, vamos configurar função de perda, otimizador e função para calcular a acurácia.

- `learning_rate` é a taxa com que o modelo ajusta os pesos — mais um hiperparâmetro.
- `num_epochs` é o número de épocas de treinamento.
- `device` define o hardware: se houver GPU disponível, usamos; caso contrário, CPU.
- `modelo` é a instância do modelo.
- `modelo.to(device)` envia o modelo para o dispositivo escolhido.
- `criterion` é a métrica de perda usada para otimizar os pesos no forward / backward.
- `optimizer` é a técnica de otimização (aqui SGD); recebe `learning_rate` e os parâmetros do modelo.


In [ ]:
learning_rate = 0.001
num_epochs = 5

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model = MyModel()
model = model.to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

A função utilitária abaixo ajuda a calcular a acurácia do modelo. Por ora não é importante entender em detalhe — basicamente ela compara as predições com os alvos verdadeiros e tira a média de acertos.


In [ ]:
## utility function to compute accuracy
def get_accuracy(output, target, batch_size):
    ''' Obtain accuracy for training round '''
    corrects = (torch.max(output, 1)[1].view(target.size()).data == target.data).sum()
    accuracy = 100.0 * corrects/batch_size
    return accuracy.item()

## Treinando o Modelo

Vamos treinar. O código a seguir pode ser descrito assim:

- O primeiro passo é definir o loop de treinamento:

```python
for epoca in range(num_epochs):
    ...
```

- Definimos duas variáveis, `training_running_loss` e `train_acc`, que ajudam a monitorar perda e acurácia ao longo dos batches.
- `modelo.treinar()` indica explicitamente que vamos treinar.
- Iteramos sobre o `dataloader`, que entrega batches em pares (imagem, label).
- O segundo `for` significa que, a cada época, iteramos sobre todos os batches.
- Alimentamos o modelo com as imagens (`modelo(images)`) e obtemos as predições.
- As predições, junto com os alvos, são usadas para calcular a perda com a função definida.
- Antes de atualizar os pesos, fazemos:
  - `optimizer.zero_grad()` para zerar gradientes acumulados (evita sobrescrever indevidamente)
  - `loss.backward()` calcula os gradientes da perda em relação aos parâmetros
  - `optimizer.step()` atualiza os parâmetros
- Acumulamos perda e acurácia para acompanhar o aprendizado.


In [ ]:
## train the model
for epoch in range(num_epochs):
    train_running_loss = 0.0
    train_acc = 0.0

    ## commence training
    model = model.train()

    ## training step
    for i, (images, labels) in enumerate(trainloader):
        
        images = images.to(device)
        labels = labels.to(device)

        ## forward + backprop + loss
        predictions = model(images)
        loss = criterion(predictions, labels)
        optimizer.zero_grad()
        loss.backward()

        ## update model params
        optimizer.step()

        train_running_loss += loss.detach().item()
        train_acc += get_accuracy(predictions, labels, BATCH_SIZE)
    
    model.eval()
    print('Epoch: %d | Loss: %.4f | Train Accuracy: %.2f' \
          %(epoch, train_running_loss / i, train_acc/i)) 

Depois de todas as épocas, dá pra ver que a perda continua caindo e a acurácia de treino subindo — sinal de que o modelo está aprendendo a classificar as imagens.

Podemos verificar isso calculando a acurácia no conjunto de teste para ver como o modelo generaliza. Como você verá abaixo, esse modelo simples vai muito bem na tarefa de classificação do MNIST.


In [ ]:
test_acc = 0.0
for i, (images, labels) in enumerate(testloader, 0):
    images = images.to(device)
    labels = labels.to(device)
    outputs = model(images)
    test_acc += get_accuracy(outputs, labels, BATCH_SIZE)
        
print('Test Accuracy: %.2f'%( test_acc/i))

## Considerações Finais

Parabéns por chegar até o fim! Este foi um tutorial longo cobrindo o básico de classificação de imagens com redes neurais e PyTorch.


## References

- Series repo: [github.com/Nandobez/ml-notebooks](https://github.com/Nandobez/ml-notebooks)
- Author: [Nandobez](https://github.com/Nandobez)
